# `consilience stress`: checking scoring code before a search runs

An optimiser such as AlphaEvolve, OpenEvolve or ShinkaEvolve will exploit any loophole in its
scoring function. `consilience stress` takes the scorer plus a short contract (what a valid answer is,
which transformations must not change the score, the published record) and looks for a
counterexample *before* the search runs. Every result has a receipt that `verify` re-checks.

This notebook runs real cases from `benchmarks/evaluator-stress/`, all on published code at
pinned commits (`SOURCES.yaml`):

1. a publicly reported bug, before and after its fix (non-finite input);
2. a scorer that trusts a number the candidate reports about itself;
3. a claimed record that is within the scorer's own rounding (check C5);
4. re-checking two stored findings on DeepMind's problem notebooks without re-running the search.

What it is not: a proof (PASS means no counterexample within the budget), a security sandbox, or a
discovery engine for deep loopholes. See `docs/stress.md` and `benchmarks/evaluator-stress/results/RESULTS.md`.

In [1]:
import json, tempfile, time
from pathlib import Path
from consilience import __version__, verify
from consilience.stress import stress

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
B = ROOT / "benchmarks" / "evaluator-stress"
OUT = Path(tempfile.mkdtemp(prefix="stress_demo_"))
print("consilience", __version__)

def run(rel, budget="standard"):
    t = time.time()
    r = stress(B / rel / "contract.yaml", budget=budget, out=OUT / rel.replace("/", "_"))
    print(r.text)
    print(f"[{time.time() - t:.0f} s, exit code {r.exit_code}]")
    return r

consilience 0.3.0.dev0


## 1. A reported bug, before and after its fix

OpenEvolve issue #548: the circle-packing-in-a-rectangle evaluator accepted NaN radii. The contract
says radii are finite and non-negative; check C1 feeds invalid inputs and asks whether the scorer
accepts them and whether an optimiser could profit.

In [2]:
pre = run("contracts/LABELS/oe548_cpr_pre")

consilience stress: openevolve/circle_packing_rect (pre-fix)
BLOCK: 2 finding(s) with a counterexample you can re-run.
margin m = 5e-05 (from record_resolution); reference score 0.798873; record 1.0000
  BLOCK C1 accepts_gross_invalid [finite,packing]: attack=circles:nonfinite[0, 0]=nan, score=0.798873
  BLOCK C1 accepts_gross_invalid [finite,packing]: attack=circles:nonfinite[0, 1]=nan, score=0.798873
  C3 naive search: 8 restarts, 0% ended invalid (best invalid None, best valid 0.827591)
  note: this contract was drafted by a model; read it before trusting a PASS
  limits: naive search; not a security sandbox; findings concern the hashed scoring code only
[10 s, exit code 1]


In [3]:
post = run("contracts/LABELS/oe548_cpr_post")

consilience stress: openevolve/circle_packing_rect (post-fix)
PASS: no counterexample found within the 'standard' budget (seed 0). Not a proof.
margin m = 5e-05 (from record_resolution); reference score 0.798873; record 1.0000
  C3 naive search: 8 restarts, 0% ended invalid (best invalid None, best valid 0.827591)
  note: this contract was drafted by a model; read it before trusting a PASS
  limits: naive search; not a security sandbox; findings concern the hashed scoring code only
[14 s, exit code 0]


## 2. A scorer that trusts a self-reported value

OpenEvolve issue #562 (Erdős minimum overlap): the program returns its own claimed bound, and the
pre-fix evaluator scores that claim instead of recomputing it. Check C2 perturbs only the reported
value and watches the score move.

In [4]:
erdos = run("contracts/LABELS/oe562_erdos_pre")

consilience stress: openevolve/erdos_min_overlap (pre-fix)
BLOCK: 1 finding(s) with a counterexample you can re-run.
margin m = 5e-05 (from record_resolution); reference score 0.761846; record 1.0000
  BLOCK C2 trusts_self_report: score_gain=0.00016002, slack_claim_units=0.000104999
  reported (not blocking): C1:accepts_invalid_low_score, C1:tolerance_headroom, C1-escalation:accepts_invalid_low_score
  C3 naive search: 8 restarts, 100% ended invalid (best invalid 0.768227, best valid None)
  note: this contract was drafted by a model; read it before trusting a PASS
  limits: naive search; not a security sandbox; findings concern the hashed scoring code only
[11 s, exit code 1]


## 3. A claimed record within rounding (C5)

From the pre-lock pilot: a float "counterexample" to a matrix AM-GM inequality scores +1.9e-6 under
the published `get_score`, but is negative at 60 digits. As a claim it does not clear the scorer's
own invariance noise, so C5 rates it `claim_within_rounding`. (The tool did not find this candidate
by search; it was supplied as the claim.)

In [5]:
amgm = run("pilot/dm_amgm")

consilience stress: deepmind/matrix_multiplication_amgm get_score
PASS: no counterexample found within the 'standard' budget (seed 0). Not a proof.
margin m = 1e-09 (from default_1e-9_relative); reference score -1.01009; record 0
  C5 claimed record: claim_within_rounding (orbit -9.53674e-07 .. 1.90735e-06, record 0)
  reported (not blocking): C1:accepts_invalid_low_score, C1-escalation:accepts_invalid_low_score
  C3 naive search: 8 restarts, 0% ended invalid (best invalid None, best valid -3.63044e-06)
  limits: naive search; not a security sandbox; findings concern the hashed scoring code only
[13 s, exit code 0]


## 4. Re-checking stored findings without re-running the search

`verify` recomputes each BLOCK's deciding inequality from the receipt, the stored candidate and the
hashed scorer. These two concern DeepMind's AlphaEvolve problem notebooks (commit 8f44745):

- `flat_polynomials` C⁻: the coefficients are not checked to be ±1, and scaling them improves the score;
- `equidistant_points_in_convex_polygons` at the `large` budget: a wrong vertex count returns +1e9, and
  non-convex point sets score better than AlphaEvolve's construction.

In [6]:
for rel in ["results/receipts/DM/flat_polynomials_c_minus/receipt.json",
            "results/extra/receipts/DM/equidistant_points_in_convex_polygons_large/receipt.json"]:
    v = verify(B / rel)
    print(rel, "->", "OK" if v.ok else "FAILED")
    for name, ok, note in v.steps:
        print(f"   {'ok ' if ok else 'BAD'} {name}: {note}")

results/receipts/DM/flat_polynomials_c_minus/receipt.json -> OK
   ok  receipt hash: matches its contents
   ok  contract hash: same contract
   ok  scorer and input files: same bytes
   ok  finding 0: C1 exploitable_type_violation: score 5.173792369321692e+149 vs max(projected 7.006009288212139e-06, reference 0.517379236932169) + m 1e-09
   ok  finding 1: C1 exploitable_type_violation: score 5.17379236932169e+299 vs max(projected 7.006009288212139e-06, reference 0.517379236932169) + m 1e-09
   ok  finding 2: C1 exploitable_type_violation: score 0.5173797543114057 vs max(projected 0.517379236932169, reference 0.517379236932169) + m 1e-09
   ok  finding 3: C1 exploitable_type_violation: score 0.5174309748558616 vs max(projected 0.517379236932169, reference 0.517379236932169) + m 1e-09
   ok  finding 4: C1 exploitable_type_violation: score 0.5178966161691011 vs max(projected 0.517379236932169, reference 0.517379236932169) + m 1e-09
   ok  finding 5: C1 exploitable_type_violation: score 0

results/extra/receipts/DM/equidistant_points_in_convex_polygons_large/receipt.json -> OK
   ok  receipt hash: matches its contents
   ok  contract hash: same contract
   ok  scorer and input files: same bytes
   ok  finding 0: C1 sentinel_is_best: rejected with raw 1000000000.0; better than the reference -0.49011716212553025
   ok  finding 1: C1 sentinel_is_best: rejected with raw 1000000000.0; better than the reference -0.49011716212553025
   ok  finding 2: C1 sentinel_is_best: rejected with raw 1000000000.0; better than the reference -0.49011716212553025
   ok  finding 3: C3 exploitable_invalid: worst -0.24436096277300873 vs max(record, valid base) + m 5e-06


## 5. Summary of this notebook's runs

In [7]:
for name, r in [("oe548 pre", pre), ("oe548 post", post), ("oe562 pre", erdos), ("AM-GM claim", amgm)]:
    print(f"{name:12s} {r.verdict}")
print("receipts written under", OUT)

oe548 pre    BLOCK
oe548 post   PASS
oe562 pre    BLOCK
AM-GM claim  PASS
receipts written under /tmp/stress_demo_l62f8ybr
